In [1]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

In [2]:
INPUT_CSV = "../../data/cleaned/districts_w_coords.csv"
OUTPUT_CSV = "../../data/cleaned/unique_districts_pvout.csv"


In [3]:
async def get_pvout(page, lat, lon):
    url = f"https://globalsolaratlas.info/map?c={lat},{lon},6&s={lat},{lon}&m=site"
    await page.goto(url, wait_until="networkidle", timeout=60000)
    try:
        await page.wait_for_selector("gsa-site-data-item", timeout=20000)
        items = await page.query_selector_all("gsa-site-data-item")
        for item in items:
            key_el = await item.query_selector("gsa-site-data-key")
            if key_el and "PVOUT" in await key_el.inner_text():
                value_el = await item.query_selector("sg-unit-value-inner")
                if value_el:
                    return float((await value_el.inner_text()).strip().split("\n")[0])
    except Exception as e:
        print(f"  Error ({lat}, {lon}): {e}")
    return None

async def main():
    # Leer el CSV original y obtener distritos únicos
    df_full = pd.read_csv(INPUT_CSV)
    df = df_full.drop_duplicates(subset=['DISTRITO'])[['DISTRITO', 'LATITUD', 'LONGITUD']].copy()
    df.reset_index(drop=True, inplace=True)
    df["PVOUT"] = None

    print(f"Se procesarán {len(df)} distritos únicos.")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for i, row in df.iterrows():
            lat, lon = row["LATITUD"], row["LONGITUD"]
            distrito = row["DISTRITO"]
            print(f"[{i+1}/{len(df)}] {distrito} ({lat}, {lon})")
            pvout = await get_pvout(page, lat, lon)
            df.at[i, "PVOUT"] = pvout
            # Guardar progreso
            if (i + 1) % 10 == 0:
                df.to_csv(OUTPUT_CSV, index=False)
            print(f"  {'✓ ' + str(pvout) + ' kWh/kWp' if pvout else '✗ Sin resultado'}")
            await asyncio.sleep(2)

        await browser.close()

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nGuardado en {OUTPUT_CSV}")
    print(df.head())

import nest_asyncio
nest_asyncio.apply()
asyncio.run(main())

Se procesarán 109 distritos únicos.
[1/109] ACARI (-15.4325, -74.617222)
  ✓ 1729.1 kWh/kWp
[2/109] ACHOMA (-15.661111, -71.701667)
  ✓ 2016.8 kWh/kWp
[3/109] ALCA (-15.134167, -72.764722)
  ✓ 1777.6 kWh/kWp
[4/109] ALTO SELVA ALEGRE (-16.370556, -71.527222)
  ✓ 2026.3 kWh/kWp
[5/109] ANDAGUA (-15.4975, -72.355)
  ✓ 2040.2 kWh/kWp
[6/109] ANDARAY (-15.796111, -72.859722)
  ✓ 2042.9 kWh/kWp
[7/109] APLAO (-16.076111, -72.4925)
  ✓ 1847.4 kWh/kWp
[8/109] AREQUIPA (-16.400833, -71.537778)
  ✓ 2030.6 kWh/kWp
[9/109] ATICO (-16.208889, -73.625833)
  ✓ 1684.0 kWh/kWp
[10/109] ATIQUIPA (-15.795556, -74.365833)
  ✓ 1612.1 kWh/kWp
[11/109] AYO (-15.683611, -72.274444)
  ✓ 1951.4 kWh/kWp
[12/109] BELLA UNION (-15.451944, -74.662222)
  ✓ 1728.0 kWh/kWp
[13/109] CABANACONDE (-15.621389, -71.979722)
  ✓ 1999.4 kWh/kWp
[14/109] CAHUACHO (-15.504167, -73.481667)
  ✓ 2058.0 kWh/kWp
[15/109] CALLALLI (-15.506667, -71.448333)
  ✓ 1998.5 kWh/kWp
[16/109] CAMANA (-16.623611, -72.711389)
  ✓ 1637.5 kWh/kWp